In [8]:
import sys, os, json, sqlite3
import pandas as pd
from datetime import datetime

sys.path.append(os.path.expanduser('~/Documents/IA/4_semestre/ing_de_datos/ing_datos/tools'))
import tools

VERSION_TRANSFORM = "v1"

for p in (tools.RUTA_TRANSF_PROCESSED, tools.RUTA_TRANSF_LOGS, tools.RUTA_TRANSF_REPORTES):
    p.mkdir(parents=True, exist_ok=True)

print("Datasets:", tools.ETIQUETAS)

Datasets: ['adilshamim8', 'samithsachidanandan', 'saurabhbadole']


In [9]:
def transformar(etiqueta):
    cfg             = tools.DATASETS[etiqueta]
    db_path         = cfg["db"]
    tabla_extracted = f"extracted_{etiqueta}"
    nombre_destino  = f"transformed_{etiqueta}"
    parquet         = tools.RUTA_TRANSF_PROCESSED / f"{nombre_destino}.parquet"
    lineage         = tools.RUTA_TRANSF_LOGS / f"lineage_{nombre_destino}.json"

    try:
        # 1. Leer EXTRACTED (solo lectura)
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        df_ext = pd.read_sql_query(f"SELECT * FROM {tabla_extracted}", conn)
        conn.close()

        tools.logs.escribir_log(tools.LOG_TRANSF, etiqueta=etiqueta, etapa="lectura_extracted",
                                origen=tabla_extracted, filas_entrada=len(df_ext), estado="exitoso")

        # 2. TRANSFORMACIONES: aquí va el código de tu equipo.
        #    Debe terminar con df_final y bitacora definidos.
        #    (Mientras tanto, esto solo copia los datos para poder probar los logs.)
        df_final = df_ext.copy()
        bitacora = []

        # 3. Carga a Parquet
        df_final.to_parquet(parquet, index=False)
        tools.logs.escribir_log(tools.LOG_TRANSF, etiqueta=etiqueta, etapa="carga_parquet",
                                origen=tabla_extracted, destino=str(parquet),
                                filas_entrada=len(df_ext), filas_salida=len(df_final),
                                version=VERSION_TRANSFORM, estado="exitoso")

        # 4. Releer el Parquet y comprobar tipos
        df_leido = pd.read_parquet(parquet)
        tipos_ok = df_leido.dtypes.astype(str).to_dict() == df_final.dtypes.astype(str).to_dict()
        tools.logs.escribir_log(tools.LOG_TRANSF, etiqueta=etiqueta, etapa="lectura_validacion",
                                filas_entrada=len(df_final), filas_salida=len(df_leido),
                                tipos_conservados=tipos_ok, estado="exitoso" if tipos_ok else "error")

        # 5. Lineage (siempre se sobrescribe)
        tools.logs.crear_cdc(
            lineage, etiqueta, reemplazar=True,
            recorrido=["Kaggle (CSV)", f"{cfg['raw']} ({db_path.name})",
                       f"{tabla_extracted} ({db_path.name})", f"{nombre_destino}.parquet"],
            origen=tabla_extracted, destino=str(parquet),
            filas_origen=len(df_ext), filas_destino=len(df_final),
            version_reglas=VERSION_TRANSFORM, transformaciones=bitacora,
            columnas_destino=df_final.dtypes.astype(str).to_dict(),
        )

        print(f"[{etiqueta}] {len(df_ext):,} → {len(df_final):,} filas | tipos conservados: {tipos_ok}")

    except Exception as e:
        tools.logs.escribir_log(tools.LOG_TRANSF, etiqueta=etiqueta, etapa="error",
                                estado="error", error=str(e))
        print(f"[{etiqueta}] ERROR: {e}")
        raise

In [10]:
for etiqueta in tools.ETIQUETAS:
    transformar(etiqueta)

[adilshamim8] 151,445 → 151,445 filas | tipos conservados: True
[samithsachidanandan] 88,584 → 88,584 filas | tipos conservados: True
[saurabhbadole] 93,597 → 93,597 filas | tipos conservados: True


In [11]:
log = pd.read_json(tools.LOG_TRANSF, lines=True)
display(log[["timestamp", "etiqueta", "etapa", "filas_entrada", "filas_salida", "estado"]])

for et in tools.ETIQUETAS:
    print(et,
          "| parquet:", (tools.RUTA_TRANSF_PROCESSED / f"transformed_{et}.parquet").exists(),
          "| lineage:", (tools.RUTA_TRANSF_LOGS / f"lineage_transformed_{et}.json").exists())

,timestamp,etiqueta,etapa,filas_entrada,filas_salida,estado
0,2026-11-07 19:05:41,adilshamim8,lectura_extracted,151445,NaN,exitoso
1,2026-11-07 19:05:41,adilshamim8,carga_parquet,151445,151445.0,exitoso
2,2026-11-07 19:05:41,adilshamim8,lectura_validacion,151445,151445.0,exitoso
3,2026-11-07 19:05:41,samithsachidanandan,lectura_extracted,88584,NaN,exitoso
4,2026-11-07 19:05:41,samithsachidanandan,carga_parquet,88584,88584.0,exitoso
5,2026-11-07 19:05:41,samithsachidanandan,lectura_validacion,88584,88584.0,exitoso
6,2026-11-07 19:05:42,saurabhbadole,lectura_extracted,93597,NaN,exitoso
7,2026-11-07 19:05:42,saurabhbadole,carga_parquet,93597,93597.0,exitoso
8,2026-11-07 19:05:42,saurabhbadole,lectura_validacion,93597,93597.0,exitoso


adilshamim8 | parquet: True | lineage: True
samithsachidanandan | parquet: True | lineage: True
saurabhbadole | parquet: True | lineage: True
